# Fixture-level ensemble model

Ensemble model that takes in 2024/25 fixture data along with the pre-season and in-season models


In [10]:
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from prediction.model.pre_season_model import pre_season_model
from prediction.model.in_season_model import in_season_model
from training.load_training_data import load_historic_player_fixture_data, load_fixtures_data
from training.config import RANDOM_STATE, NUMERIC_FEATURES, TARGET_COLUMN

# Load 2024/25 fixture data

In [11]:
TRAINING_SEASONS = ["2024-25"]

fixture_history_df = pd.concat(
    [
        load_historic_player_fixture_data(season)
        .rename(columns={"GW": "target_gw"})
        .assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
)

fixture_history_df = fixture_history_df.sort_values(
    ["player_id", "season", "target_gw", "fixture_id"]
).reset_index(drop=True)

fixture_history_df = fixture_history_df[
            [
                "player_id",
                "name",
                "season",
                "target_gw",
                "fixture_id",
                "was_home"
            ] + NUMERIC_FEATURES + [TARGET_COLUMN]

]

print(fixture_history_df.groupby("season").size())
fixture_history_df.head()

season
2024-25    27605
dtype: int64


,player_id,name,season,target_gw,fixture_id,was_home,total_points,minutes,goals_scored,assists,...,influence,creativity,threat,ict_index,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points
0,1,Fábio Ferreira Vieira,2024-25,1,2,True,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0
1,1,Fábio Ferreira Vieira,2024-25,2,11,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0
2,1,Fábio Ferreira Vieira,2024-25,3,21,True,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0
3,1,Fábio Ferreira Vieira,2024-25,4,39,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0
4,1,Fábio Ferreira Vieira,2024-25,5,47,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0


### Add in fixture details

In [12]:
fixture_details_df = load_fixtures_data('2024-25')[[
    'id', 'team_a_difficulty', 'team_h_difficulty'
]].rename(columns = {'id': 'fixture_id'})

fixture_history_df = fixture_history_df.merge(
    fixture_details_df,
    on = "fixture_id"
)

fixture_history_df['player_game_difficulty'] = np.where(
    fixture_history_df['was_home'], 
    fixture_history_df['team_h_difficulty'], 
    fixture_history_df['team_a_difficulty']
    )

fixture_history_df['opponent_game_difficulty'] = np.where(
    ~fixture_history_df['was_home'], 
    fixture_history_df['team_h_difficulty'], 
    fixture_history_df['team_a_difficulty']
    )


fixture_history_df

,player_id,name,season,target_gw,fixture_id,was_home,total_points,minutes,goals_scored,assists,...,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,team_a_difficulty,team_h_difficulty,player_game_difficulty,opponent_game_difficulty
0,1,Fábio Ferreira Vieira,2024-25,1,2,True,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,5,3,3,5
1,1,Fábio Ferreira Vieira,2024-25,2,11,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,4,5,4,5
2,1,Fábio Ferreira Vieira,2024-25,3,21,True,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,5,3,3,5
3,1,Fábio Ferreira Vieira,2024-25,4,39,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,3,5,3,5
4,1,Fábio Ferreira Vieira,2024-25,5,47,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,4,5,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27600,801,Brayden Clarke,2024-25,37,361,True,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,5,4,4,5
27601,801,Brayden Clarke,2024-25,38,378,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,1,5,1,5
27602,802,Sammy Braybrooke,2024-25,38,371,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,3,1,3,1
27603,803,Reece Welch,2024-25,38,376,False,0,0,0,0,...,0,0.0,0.0,0.0,0.0,0,4,3,4,3


### Get pre-season model scores 

In [13]:
TRAINING_SEASONS = ["2023-24"]

pre_season_history = pd.concat(
    [
        load_historic_player_fixture_data(season).assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
)

pre_season_history = pre_season_history.groupby(["name","position"])[NUMERIC_FEATURES].sum(min_count=1).rename(columns={
        feature: f"season_sum_{feature}" for feature in NUMERIC_FEATURES
    }).reset_index()


pre_season_history['pre_season_model_score'] = pre_season_model.predict_for_dataframe(pre_season_history)

fixture_history_df = fixture_history_df.merge(
    pre_season_history[["name", "pre_season_model_score"]],
    on = ['name'],
    how = 'left'
)

fixture_history_df

,player_id,name,season,target_gw,fixture_id,was_home,total_points,minutes,goals_scored,assists,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,team_a_difficulty,team_h_difficulty,player_game_difficulty,opponent_game_difficulty,pre_season_model_score
0,1,Fábio Ferreira Vieira,2024-25,1,2,True,0,0,0,0,...,0.0,0.0,0.0,0.0,0,5,3,3,5,0.663651
1,1,Fábio Ferreira Vieira,2024-25,2,11,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,4,5,4,5,0.663651
2,1,Fábio Ferreira Vieira,2024-25,3,21,True,0,0,0,0,...,0.0,0.0,0.0,0.0,0,5,3,3,5,0.663651
3,1,Fábio Ferreira Vieira,2024-25,4,39,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,3,5,3,5,0.663651
4,1,Fábio Ferreira Vieira,2024-25,5,47,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,4,5,4,5,0.663651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27600,801,Brayden Clarke,2024-25,37,361,True,0,0,0,0,...,0.0,0.0,0.0,0.0,0,5,4,4,5,NaN
27601,801,Brayden Clarke,2024-25,38,378,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,1,5,1,5,NaN
27602,802,Sammy Braybrooke,2024-25,38,371,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,3,1,3,1,NaN
27603,803,Reece Welch,2024-25,38,376,False,0,0,0,0,...,0.0,0.0,0.0,0.0,0,4,3,4,3,NaN


### In-season model scores

In [14]:
fixture_history_df['in_season_model_score'] = in_season_model.predict_for_dataframe(fixture_history_df)


ValueError: Provided DataFrame is missing required columns: ['position', 'current_gw', 'horizon', 'calc_total_points_mean_last_1_fixtures', 'calc_minutes_mean_last_1_fixtures', 'calc_goals_scored_mean_last_1_fixtures', 'calc_assists_mean_last_1_fixtures', 'calc_clean_sheets_mean_last_1_fixtures', 'calc_goals_conceded_mean_last_1_fixtures', 'calc_own_goals_mean_last_1_fixtures', 'calc_penalties_saved_mean_last_1_fixtures', 'calc_penalties_missed_mean_last_1_fixtures', 'calc_yellow_cards_mean_last_1_fixtures', 'calc_red_cards_mean_last_1_fixtures', 'calc_saves_mean_last_1_fixtures', 'calc_bonus_mean_last_1_fixtures', 'calc_bps_mean_last_1_fixtures', 'calc_influence_mean_last_1_fixtures', 'calc_creativity_mean_last_1_fixtures', 'calc_threat_mean_last_1_fixtures', 'calc_ict_index_mean_last_1_fixtures', 'calc_starts_mean_last_1_fixtures', 'calc_expected_goals_mean_last_1_fixtures', 'calc_expected_assists_mean_last_1_fixtures', 'calc_expected_goal_involvements_mean_last_1_fixtures', 'calc_ex

In [ ]:
    "target_gw",
    "horizon",
    "position",
    "was_home"
    "player_game_difficulty"
    "opponent_game_difficulty"
    "pre_season_model_score",
    "in_season_model_score",

('inseason_model_score',)

## Leakage-free base-model scores

The base-model notebooks own their historical training periods. This notebook loads their exported scores/models and only builds ensemble rows from the 2025/26 fixture data.


In [ ]:
season_2025_26 = (
    load_historic_player_fixture_data("2025-26")
    .drop_duplicates(["player_id", "fixture"], keep="last")
    .reset_index(drop=True)
)
print("2025/26:", season_2025_26.shape)


In [ ]:
preseason_scores = load_preseason_scores()
preseason_scores.head()


In [ ]:
inseason_scores = load_inseason_scores()
inseason_scores.head()


## Ensemble dataset


In [ ]:
team_lookup = load_team_data("2025-26").set_index("id")["name"]
player_codes = load_player_data("2025-26")[["id", "code"]].rename(columns={"id": "player_id"})
ensemble_data = season_2025_26.merge(
    inseason_scores[["player_id", "fixture", "inseason_model_score"]],
    on=["player_id", "fixture"], how="left", validate="one_to_one",
)
ensemble_data = ensemble_data.merge(player_codes, on="player_id", how="left")
ensemble_data = ensemble_data.merge(
    preseason_scores[["code", "preseason_model_score"]], on="code", how="left"
)
ensemble_data["opponent"] = ensemble_data["opponent_team"].map(team_lookup)
ensemble_data["fixtures_in_gameweek"] = ensemble_data.groupby(
    ["player_id", "GW"]
)["fixture"].transform("nunique")
ensemble_data["is_double_gameweek"] = ensemble_data["fixtures_in_gameweek"].gt(1).astype(int)
ensemble_data["was_home"] = ensemble_data["was_home"].astype(int)
promoted_teams = get_promoted_teams("2025-26")
ensemble_data["team_promoted"] = ensemble_data["team"].isin(promoted_teams).astype(int)
ensemble_data["opponent_promoted"] = ensemble_data["opponent"].isin(promoted_teams).astype(int)

ensemble_features = list(ENSEMBLE_FEATURES)
categorical_features = list(ENSEMBLE_CATEGORICAL_FEATURES)
target = "total_points"
ensemble_data[categorical_features] = ensemble_data[categorical_features].fillna("__MISSING__").astype(str)
assert not ensemble_data.duplicated(["player_id", "fixture"]).any()
assert ensemble_data["opponent"].ne("__MISSING__").all()
print(f"Fixture rows: {len(ensemble_data):,}")
ensemble_data[["name", "fixture", *ensemble_features, target]].head()


In [ ]:
def fit_ensemble_model(training_data):
    model = CatBoostRegressor(
        iterations=250, learning_rate=0.03, depth=6, loss_function="RMSE",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
    )
    model.fit(
        training_data[ensemble_features], training_data[target],
        cat_features=categorical_features,
    )
    return model


def evaluate(validation, predictions, top_fraction=0.10):
    scored = validation[["GW", target]].copy()
    scored["prediction"] = predictions
    scored["absolute_error"] = (scored[target] - scored["prediction"]).abs()
    top_end_by_gameweek = []
    for _, gameweek in scored.groupby("GW"):
        n = max(1, int(np.ceil(len(gameweek) * top_fraction)))
        predicted_top = gameweek.nlargest(n, "prediction")
        actual_top_indices = set(gameweek.nlargest(n, target).index)
        top_end_by_gameweek.append({
            "top_decile_avg_actual_points": predicted_top[target].mean(),
            "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
            "top_decile_oracle_regret": (
                gameweek.nlargest(n, target)[target].mean()
                - predicted_top[target].mean()
            ),
        })
    top_end = pd.DataFrame(top_end_by_gameweek).mean()
    return {
        "num": len(scored),
        "MAE": mean_absolute_error(scored[target], scored["prediction"]),
        "RMSE": root_mean_squared_error(scored[target], scored["prediction"]),
        "R2": r2_score(scored[target], scored["prediction"]),
        **top_end.to_dict(),
        **{
            f"absolute_error_p{percentile}": scored["absolute_error"].quantile(percentile / 100)
            for percentile in (25, 50, 75, 90)
        },
    }


## Single late-season holdout: GW30+

Train on GW1–29 and evaluate every fixture from GW30 onward. This provides one conventional holdout alongside the phase-by-phase walk-forward results below.


In [ ]:
holdout_train = ensemble_data.loc[ensemble_data["GW"].lt(30)]
holdout_validation = ensemble_data.loc[ensemble_data["GW"].ge(30)]
holdout_model = fit_ensemble_model(holdout_train)
holdout_predictions = holdout_model.predict(holdout_validation[ensemble_features])
holdout_results = pd.DataFrame([
    {"train_gws": "1-29", "validation_gws": "30-38",
     **evaluate(holdout_validation, holdout_predictions)}
])
holdout_results.T.round(3)


## Chronological phase validation

Every fold trains on earlier gameweeks only. GW1–5 provide the minimum fitting history, so honest validation of GW1–5 itself would require ensemble training data from a previous season.


In [ ]:
folds = [
    ("early", 5, 6, 10),
    ("early-mid", 10, 11, 15),
    ("mid", 19, 20, 24),
    ("late-mid", 28, 29, 33),
    ("late", 33, 34, 38),
]

results = []
validation_predictions = []
for phase, train_end, valid_start, valid_end in folds:
    train = ensemble_data.loc[ensemble_data["GW"].le(train_end)]
    validation = ensemble_data.loc[ensemble_data["GW"].between(valid_start, valid_end)]
    model = fit_ensemble_model(train)
    predictions = model.predict(validation[ensemble_features])
    results.append({
        "phase": phase, "train_through": train_end,
        "validation_gws": f"{valid_start}-{valid_end}",
        **evaluate(validation, predictions),
    })
    validation_predictions.append(
        validation[["player_id", "fixture", "GW", target]].assign(phase=phase, prediction=predictions)
    )

fold_results = pd.DataFrame(results)
fold_results.round(3)


In [ ]:
out_of_time_predictions = pd.concat(validation_predictions, ignore_index=True)
pd.Series(
    evaluate(out_of_time_predictions, out_of_time_predictions["prediction"]),
    name="all chronological folds",
).round(3)


## Production ensemble artifact

After validation, refit the same ensemble procedure on every available 2025/26 fixture row. This artifact is separate from the fold models above and is used only to forecast later seasons.


In [ ]:
final_ensemble_model = fit_ensemble_model(ensemble_data)
DEFAULT_ENSEMBLE_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": final_ensemble_model,
        "feature_columns": ensemble_features,
        "categorical_columns": categorical_features,
        "training_season": "2025-26",
    },
    DEFAULT_ENSEMBLE_ARTIFACT_PATH,
)
print(f"Saved production ensemble model to {DEFAULT_ENSEMBLE_ARTIFACT_PATH}")
